# Seminar 10: Diffusion Probabilistic Models

Notebook is based on https://github.com/acids-ircam/diffusion_models/.

Date: 2025-03-18

## Notebook Structure
The notebook covers both the original diffusion formulation and the DDPM (Denoising Diffusion Probabilistic Models) approach.

## Setup and Helper Functions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.datasets import make_swiss_roll

def hdr_plot_style():
    plt.style.use('dark_background')
    plt.rcParams.update({
        'font.size': 18,
        'lines.linewidth': 3,
        'lines.markersize': 15,
        'ps.useafm': True,
        'pdf.use14corefonts': True,
        'text.usetex': False,
        'font.family': 'sans-serif',
        'font.sans-serif': 'Courier New'
    })

hdr_plot_style()

# Set device
cuda = torch.cuda.is_available()
device = torch.device("cuda" if cuda else ("mps" if torch.backends.mps.is_available() else "cpu"))
torch.backends.cudnn.benchmark = cuda

print(f"Using device: {device}")

### Data Sampling and Visualization

We use the classic Swiss roll dataset (projected to 2D) for all our demonstrations. You can tweak the noise level via the `noise` parameter.

In [ ]:
def sample_swiss_roll_batch(batch_size: int, noise: float = 0.5) -> np.ndarray:
    """
    Generate a batch of 2D points from a Swiss roll dataset.
    
    Args:
        batch_size (int): Number of points.
        noise (float): Noise level.
    
    Returns:
        np.ndarray: 2D data points.
    """
    x, _ = make_swiss_roll(batch_size, noise=noise)
    return x[:, [0, 2]] / 10.0  # Use two dimensions and normalize

In [ ]:
# Sample and plot the data
data = sample_swiss_roll_batch(10**4).T
plt.figure(figsize=(16, 12))
plt.scatter(data[0], data[1], alpha=0.5, color='red', edgecolor='white', s=40)
plt.title("Swiss Roll Data")
plt.show()

## Diffusion Probabilistic Models

*Diffusion probabilistic models* are generative models characterized by a chain of latent variables $\mathbf{x}_1,\dots,\mathbf{x}_T$, each sharing the same dimensionality as the original input data $\mathbf{x}_0 \sim q(\mathbf{x}_0)$. The structure involves two core Markov processes:

- **Forward (diffusion) process**: Gradually converts the data distribution $q(\mathbf{x}_0)$ into a simpler, analytically tractable prior distribution $\pi(\mathbf{x}_T)$ through repeated stochastic transformations.
- **Reverse (parametric) process**: Learns to invert the forward diffusion, progressively reconstructing data from the simple distribution.


### Forward Diffusion Process: Gradual Noise Injection

In the forward process, the data distribution $q(\mathbf{x}_{0})$ is gradually converted into an analytically tractable distribution $\pi(\mathbf{y})$, by repeated application of a Markov diffusion kernel $T_{\pi}(\mathbf{y}\mid\mathbf{y}';\beta)$, with a given diffusion rate $\beta$. 

$$
q(\mathbf{x}_{t}\mid\mathbf{x}_{t-1}) = T_{\pi}(\mathbf{x}_{t}\mid\mathbf{x}_{t-1}; \beta_{t}), \quad t = 1, \dots, T
$$

The complete distribution $q(\mathbf{x}_{0:T})$ is called the _diffusion_ process and is defined as
$$
\begin{align*}
q(\mathbf{x}_{0:T}) &=  q(\mathbf{x}_{0}) \prod_{t=1}^{T} q(\mathbf{x}_{t}\mid\mathbf{x}_{t-1}) \\
&= q(\mathbf{x}_0)\prod_{t=1}^{T}T_{\pi}(\mathbf{x}_t\mid\mathbf{x}_{t-1};\beta_t)
\end{align*}
$$

A particularly practical instantiation of this framework occurs when selecting the kernel $T_{\pi}$ as Gaussian distributions. Explicitly, defining the diffusion kernel as:

$$
q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = \mathcal{N}\left(\mathbf{x}_t;\sqrt{1-\beta_t}\mathbf{x}_{t-1}, \beta_t\mathbf{I}\right),\quad t = 1,\dots,T
$$

This choice yields a forward process that analytically simplifies into a Gaussian distribution over the latent states.

In [ ]:
def forward_process(x_start: torch.Tensor, n_steps: int, betas: torch.Tensor) -> list:
    """
    Execute the forward (diffusion) process by gradually injecting Gaussian noise.
    
    Args:
        x_start (torch.Tensor): Initial data.
        n_steps (int): Number of diffusion steps.
        betas (torch.Tensor): Variance schedule.
    
    Returns:
        List[torch.Tensor]: Sequence of diffused samples.
    """
    x_seq = [x_start]
    for n in range(n_steps):
        noise = torch.randn_like(x_start).to(x_start.device)
        # According to q(x_t|x_{t-1}) = N(sqrt(1-beta_t)*x_{t-1}, beta_t*I)
        x_next = torch.sqrt(1 - betas[n]) * x_seq[-1] + torch.sqrt(betas[n]) * noise
        x_seq.append(x_next)
    return x_seq

In [ ]:
n_steps = 100
betas = torch.tensor([0.0005] * n_steps, device=device)
dataset = torch.tensor(data.T, dtype=torch.float32, device=device)
x_seq = forward_process(dataset, n_steps, betas)

fig, axs = plt.subplots(1, 10, figsize=(28, 3))
for i in range(10):
    idx = int((i / 10.0) * n_steps)
    axs[i].scatter(x_seq[idx][:, 0].cpu(), x_seq[idx][:, 1].cpu(), s=10)
    axs[i].set_axis_off()
    axs[i].set_title(f"$q(\\mathbf{{x}}_{{{idx}}})$")
plt.tight_layout()
plt.show()

We can define any type of variance schedules for $\beta_{1},\cdots,\beta_{n}$, as provided in the following function

In [ ]:
def make_beta_schedule(schedule: str = 'linear', n_timesteps: int = 1000,
                       start: float = 1e-5, end: float = 1e-2) -> torch.Tensor:
    """
    Create a beta (variance) schedule.
    
    Args:
        schedule (str): Type of schedule ('linear', 'quad', or 'sigmoid').
        n_timesteps (int): Number of timesteps.
        start (float): Initial beta value.
        end (float): Final beta value.
    
    Returns:
        torch.Tensor: Tensor of beta values.
    """
    if schedule == 'linear':
        betas = torch.linspace(start, end, n_timesteps, device=device)
    elif schedule == "quad":
        betas = torch.linspace(start ** 0.5, end ** 0.5, n_timesteps, device=device) ** 2
    elif schedule == "sigmoid":
        betas = torch.linspace(-6, 6, n_timesteps, device=device)
        betas = torch.sigmoid(betas) * (end - start) + start
    else:
        raise ValueError("Unsupported schedule type.")
    return betas

### Efficient Sampling at Arbitrary Timesteps

The forward diffusion process allows direct sampling of latent variables $\mathbf{x}_t$ at any arbitrary timestep $t$ from the original data $\mathbf{x}_0$ without explicitly performing all intermediate steps. Using the definitions:

- $\alpha_t = 1 - \beta_t$
- $\bar{\alpha}_t = \prod_{s=1}^{t}\alpha_s$

we have the **generalization** for directly sampling from $q(\mathbf{x}_t|\mathbf{x}_0)$:

$$
q(\mathbf{x}_t\mid\mathbf{x}_0)=\mathcal{N}\left(\mathbf{x}_t;\sqrt{\bar{\alpha}_t}\mathbf{x}_0,(1-\bar{\alpha}_t)\mathbf{I}\right).
$$

Therefore, we can update our forward sampling function accordingly, leveraging a variance schedule $\beta_1,\dots,\beta_T$ computed beforehand.

In [ ]:
# Recompute schedules using a sigmoid schedule
betas = make_beta_schedule(schedule='sigmoid', n_timesteps=n_steps, start=1e-5, end=1e-2)
alphas = 1 - betas
alphas_prod = torch.cumprod(alphas, dim=0)
alphas_prod_prev = torch.cat([torch.tensor([1.0], device=device), alphas_prod[:-1]], dim=0)
alphas_bar_sqrt = torch.sqrt(alphas_prod)
one_minus_alphas_bar_sqrt = torch.sqrt(1 - alphas_prod)
one_minus_alphas_bar_log = torch.log(1 - alphas_prod)

This allows to perform a very efficient implementation of the forward process, where we can directly sample at any given timesteps, as shown in the following code.

In [ ]:
def extract(a: torch.Tensor, t: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """
    Extract coefficients for a given timestep and reshape to match x.
    
    Args:
        a (torch.Tensor): Coefficient tensor.
        t (torch.Tensor): Timestep indices.
        x (torch.Tensor): Reference tensor for shape.
    
    Returns:
        torch.Tensor: Reshaped coefficients.
    """
    out = a.gather(0, t.to(a.device))
    return out.view(-1, *([1] * (x.dim() - 1)))

def q_sample(x_0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor = None) -> torch.Tensor:
    """
    Sample from the forward diffusion process at timestep t.
    
    Args:
        x_0 (torch.Tensor): Original data.
        t (torch.Tensor): Timestep indices.
        noise (torch.Tensor, optional): Noise to inject; if None, standard normal noise is used.
    
    Returns:
        torch.Tensor: Diffused data at timestep t.
    """
    if noise is None:
        noise = torch.randn_like(x_0)
    return extract(alphas_bar_sqrt, t, x_0) * x_0 + extract(one_minus_alphas_bar_sqrt, t, x_0) * noise

# Visualize the diffusion process at different timesteps
fig, axs = plt.subplots(1, 10, figsize=(28, 3))
for i in range(10):
    timestep = torch.tensor([i * 10], device=device)
    q_i = q_sample(dataset, timestep)
    axs[i].scatter(q_i[:, 0].cpu(), q_i[:, 1].cpu(), s=10)
    axs[i].set_axis_off()
    axs[i].set_title(f"$q(\\mathbf{{x}}_{{{i*10}}})$")
plt.tight_layout()
plt.show()

### Posterior Computation in the Forward Process

Given the Gaussian form of forward diffusion processes, the posterior distribution $q(\mathbf{x}_{t-1}\mid\mathbf{x}_t,\mathbf{x}_0)$ is analytically tractable and explicitly defined by:

- Posterior mean:
$$
\mu_{posterior} = \frac{\beta_t \sqrt{\bar{\alpha}_{t-1}}}{1 - \bar{\alpha}_t}\mathbf{x}_0 + \frac{(1-\bar{\alpha}_{t-1})\sqrt{\alpha_t}}{1 - \bar{\alpha}_t}\mathbf{x}_t
$$
- Posterior variance:
$$
\sigma_{posterior}^2=\frac{(1-\bar{\alpha}_{t-1})\beta_t}{1-\bar{\alpha}_t}
$$


The following cell computes these parameters.

In [ ]:
posterior_mean_coef_1 = (betas * torch.sqrt(alphas_prod_prev) / (1 - alphas_prod))
posterior_mean_coef_2 = ((1 - alphas_prod_prev) * torch.sqrt(alphas) / (1 - alphas_prod))
posterior_variance = betas * (1 - alphas_prod_prev) / (1 - alphas_prod)
posterior_log_variance_clipped = torch.log(torch.cat((posterior_variance[0:1], posterior_variance[1:]), dim=0))

def q_posterior_mean_variance(x_0: torch.Tensor, x_t: torch.Tensor, t: torch.Tensor) -> tuple:
    """
    Compute the mean and log variance of the posterior q(x_{t-1}|x_t, x_0).
    
    Args:
        x_0 (torch.Tensor): Original data.
        x_t (torch.Tensor): Diffused data at timestep t.
        t (torch.Tensor): Timestep indices.
    
    Returns:
        tuple: (mean, log variance)
    """
    coef1 = extract(posterior_mean_coef_1, t, x_0)
    coef2 = extract(posterior_mean_coef_2, t, x_0)
    mean = coef1 * x_0 + coef2 * x_t
    log_var = extract(posterior_log_variance_clipped, t, x_0)
    return mean, log_var

#### Reverse Denoising Process
We define the generative distribution as a Markov chain with latent states $\mathbf{x}_T, \ldots, \mathbf{x}_0$, starting from an analytically tractable prior distribution $p\left(\mathbf{x}_T\right)=\pi\left(\mathbf{x}_T\right)$, typically chosen as a standard Gaussian:

$$
p_\theta\left(\mathbf{x}_{0: T}\right)=p\left(\mathbf{x}_T\right) \prod_{t=1}^T p_\theta\left(\mathbf{x}_{t-1} \mid \mathbf{x}_t\right)
$$


Here, the reverse process transitions $p_\theta\left(\mathbf{x}_{t-1} \mid \mathbf{x}_t\right)$ approximate the unknown true posterior distributions of the forward diffusion process $q\left(\mathbf{x}_{t-1} \mid \mathbf{x}_t\right)$.

#### Conditional Gaussian Structure (Connection to VAEs)
Each reverse transition is generally modeled as a conditional Gaussian distribution, reminiscent of the decoder in Variational Autoencoders (VAEs):

$$
p_\theta\left(\mathbf{x}_{t-1} \mid \mathbf{x}_t\right)=\mathcal{N}\left(\mathbf{x}_{t-1} ; \mu_\theta\left(\mathbf{x}_t, t\right), \mathbf{\Sigma}_\theta\left(\mathbf{x}_t, t\right)\right)
$$

where:
- The mean $\mu_\theta\left(\mathbf{x}_t, t\right)$ is typically the primary learnable parameter, predicting how to best reduce noise at step $t$.
- The covariance $\boldsymbol{\Sigma}_\theta\left(\mathbf{x}_t, t\right)$ is often simplified or chosen from pre-defined schedules (commonly as a diagonal matrix with fixed or time-dependent elements). Sometimes it's partially or fully learned, though simpler forms are predominant due to practical considerations.

Here, we show a naive implementation of this process, where we have a given `model` to infer variance. Note that this model is _shared across all time steps_ but conditionned on that said time step.

In [ ]:
class ConditionalLinear(nn.Module):
    """
    A linear layer with conditional (timestep) embeddings.
    """
    def __init__(self, num_in: int, num_out: int, n_steps: int):
        super(ConditionalLinear, self).__init__()
        self.num_out = num_out
        self.linear = nn.Linear(num_in, num_out)
        self.embed = nn.Embedding(n_steps, num_out)
        nn.init.uniform_(self.embed.weight, a=0, b=1)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        out = self.linear(x)
        gamma = self.embed(t)
        return gamma * out 
    
class ConditionalModel(nn.Module):
    """
    Conditional model for diffusion that predicts concatenated mean and log variance.
    """
    def __init__(self, n_steps: int):
        super(ConditionalModel, self).__init__()
        self.lin1 = ConditionalLinear(2, 128, n_steps)
        self.lin2 = ConditionalLinear(128, 128, n_steps)
        self.lin3 = nn.Linear(128, 4)  # Output: [mean, log_variance]
        
    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        x = F.softplus(self.lin1(x, t))
        x = F.softplus(self.lin2(x, t))
        return self.lin3(x)

# Instantiate and move model to device 
model = ConditionalModel(n_steps)
    
def p_mean_variance(model: nn.Module, x: torch.Tensor, t: torch.Tensor) -> tuple:
    """
    Predict the mean and log variance from the conditional model.
    
    Args:
        model (nn.Module): The conditional model.
        x (torch.Tensor): Input at timestep t.
        t (torch.Tensor): Timestep indices.
    
    Returns:
        tuple: (mean, log variance)
    """
    out = model(x, t)
    mean, log_var = torch.split(out, 2, dim=-1)
    return mean, log_var

As we can see, the reverse process consists in inferring the values of the mean and log variance for a given timestep. Then, once we have learned the correponding model, we can perform the denoising of any given timestep, by providing both the sample $\mathbf{x}_{t}$ at a given time step, and that time step $t$ that we can use to condition the models for $\mathbf{\mu}_{\theta}(\mathbf{x}_{t},t)$ and $\mathbf{\Sigma}_{\theta}(\mathbf{x}_{t},t)$.

In [ ]:
def p_sample(model: nn.Module, x: torch.Tensor, t: int) -> torch.Tensor:
    """
    Sample x_{t-1} from x_t using the model's predictions.
    
    Args:
        model (nn.Module): The conditional model.
        x (torch.Tensor): Current sample at timestep t.
        t (int): Current timestep.
    
    Returns:
        torch.Tensor: Sampled x_{t-1}.
    """
    t_tensor = torch.tensor([t], device=x.device).long()
    mean, log_var = p_mean_variance(model, x, t_tensor)
    noise = torch.randn_like(x)
    return mean + torch.exp(0.5 * log_var) * noise

Finally, obtaining samples from the model is given by running through the whole Markov chain in reverse, starting from the normal distribution to obtain samples from the target distribution. Note that this process can be very slow if we have a large number of steps, as we need to wait for a given $\mathbf{x}_{t}$ to infer the following $\mathbf{x}_{t-1}$

In [ ]:
def p_sample_loop(model: nn.Module, shape: tuple) -> list:
    """
    Generate a full sample by iteratively applying the reverse diffusion process.
    
    Args:
        model (nn.Module): The conditional model.
        shape (tuple): Shape of the initial noise sample.
    
    Returns:
        List[torch.Tensor]: Sequence of samples from x_T down to x_0.
    """
    with torch.no_grad():
        cur_x = torch.randn(shape, device=device)
        x_seq = [cur_x]
        for i in reversed(range(n_steps)):
            cur_x = p_sample(model, cur_x, i)
            x_seq.append(cur_x)
    return x_seq

### Model Probability (Marginal Likelihood)

The marginal likelihood of the generative diffusion model $p_{\theta}(\mathbf{x}_0)$ can be expressed by integrating over the latent variables $\mathbf{x}_{1:T}$:

$$
p_{\theta}(\mathbf{x}_0) = \int p_{\theta}(\mathbf{x}_{0:T}) \, d\mathbf{x}_{1:T}
$$

This integral is generally intractable, as it involves high-dimensional latent variables. To circumvent this difficulty, we introduce a variational inference-based approach, employing a known diffusion forward process $q(\mathbf{x}_{1:T}\mid\mathbf{x}_0)$:

$$
p_{\theta}(\mathbf{x}_0) = \int p_{\theta}(\mathbf{x}_{0:T}) \frac{q(\mathbf{x}_{1:T}\mid\mathbf{x}_0)}{q(\mathbf{x}_{1:T}\mid\mathbf{x}_0)} d\mathbf{x}_{1:T}
$$

Rearranging terms to highlight the variational inference formulation explicitly, we have:

$$
p_{\theta}(\mathbf{x}_0) = \mathbb{E}_{q(\mathbf{x}_{1:T}\mid\mathbf{x}_0)}\left[\frac{p_{\theta}(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T}\mid\mathbf{x}_0)}\right]
$$

### Training via Evidence Lower Bound (ELBO)

Since direct optimization of the marginal likelihood is intractable, we instead optimize the Evidence Lower Bound (ELBO) by applying Jensen's inequality. Specifically, we derive the negative log-likelihood upper bound:

$$
-\log p_{\theta}(\mathbf{x}_0) = -\log\mathbb{E}_{q(\mathbf{x}_{1:T}\mid\mathbf{x}_0)}\left[\frac{p_{\theta}(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T}\mid\mathbf{x}_0)}\right]
$$

Applying Jensen's inequality (exploiting the concavity of the logarithm function), we obtain the upper bound:

$$
-\log p_{\theta}(\mathbf{x}_0) \leq \mathbb{E}_{q(\mathbf{x}_{1:T}\mid\mathbf{x}_0)}\left[-\log\frac{p_{\theta}(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T}\mid\mathbf{x}_0)}\right] \triangleq \mathcal{L}(\theta)
$$

Expanding explicitly into individual terms, this ELBO becomes:

$$
\mathcal{L}(\theta) = \mathbb{E}_{q}\left[-\log p(\mathbf{x}_T) - \sum_{t=1}^{T}\log\frac{p_{\theta}(\mathbf{x}_{t-1}\mid\mathbf{x}_t)}{q(\mathbf{x}_t\mid\mathbf{x}_{t-1})}\right]
$$

Notice clearly how the ELBO separates into a prior term (for the latent variable at the last timestep) and the summation over all reverse denoising steps.


### Explicit Connection to KL Divergence and Entropy Terms (Complete ELBO)

By carefully expanding the terms, we explicitly rewrite the ELBO as a sum of KL divergences and entropy terms:

$$
\mathcal{L}(\theta) 
= \underbrace{\mathbb{E}_{q}\left[-\log p(\mathbf{x}_T)\right]}_{\text{Prior likelihood}}
+ \sum_{t=1}^{T}\underbrace{\mathbb{E}_{q}\left[D_{KL}(q(\mathbf{x}_{t-1}|\mathbf{x}_t,\mathbf{x}_0)\,\Vert\, p_{\theta}(\mathbf{x}_{t-1}|\mathbf{x}_t))\right]}_{\text{Reverse process KL divergence}}
+ \underbrace{\mathbb{E}_{q}\left[-\log q(\mathbf{x}_T|\mathbf{x}_{T-1})\right]}_{\text{Forward process entropy (constant w.r.t. }\theta)}
$$

### Explicit Decomposition

If we fully expand and explicitly include all entropy terms, the complete training loss (as in the original Sohl-Dickstein formulation) is:

$$
\mathcal{L}(\theta) 
= \sum_{t=1}^{T}\mathbb{E}_{q}\left[D_{KL}(q(\mathbf{x}_{t-1}\mid\mathbf{x}_{t},\mathbf{x}_{0})\,\Vert\, p_{\theta}(\mathbf{x}_{t-1}\mid\mathbf{x}_{t}))\right] 
+ \mathbb{E}_{q}\left[-\log p(\mathbf{x}_{T})\right]
+ H_q(\mathbf{X}_T|\mathbf{X}_0) - H_q(\mathbf{X}_1|\mathbf{X}_0) - H_p(\mathbf{X}_T)
$$

Where explicitly:

- $H_q(\mathbf{X}_T|\mathbf{X}_0)$: Entropy of the forward process at the final timestep.
- $H_q(\mathbf{X}_1|\mathbf{X}_0)$: Entropy of the forward process at the initial timestep.
- $H_p(\mathbf{X}_T)$: Entropy of the prior distribution at timestep $T$.

These entropy terms explicitly appear in theoretical analyses but are constants (with respect to $\theta$) that vanish during gradient-based training optimization.

### Practical Training Objective (Simplified):

Since entropy terms do not affect optimization, the practical training loss that is commonly used for gradient-based optimization simplifies to the KL-divergence form:

$$
\mathcal{L}_{\text{practical}}(\theta) 
= \sum_{t=1}^{T}\mathbb{E}_{q}\left[D_{KL}(q(\mathbf{x}_{t-1}\mid\mathbf{x}_t,\mathbf{x}_0)\,\Vert\, p_{\theta}(\mathbf{x}_{t-1}\mid\mathbf{x}_t))\right] + C
$$

where $C$ encapsulates all entropy terms that remain constant w.r.t. the parameters $\theta$.

In [ ]:
def normal_kl(mean1: torch.Tensor, logvar1: torch.Tensor, mean2: torch.Tensor, logvar2: torch.Tensor) -> torch.Tensor:
    """
    Compute the KL divergence between two Gaussian distributions.
    """
    kl = 0.5 * (-1.0 + logvar2 - logvar1 + torch.exp(logvar1 - logvar2) +
                ((mean1 - mean2) ** 2) * torch.exp(-logvar2))
    return kl

def entropy(val: torch.Tensor) -> torch.Tensor:
    """
    Compute the entropy of a Gaussian with variance 'val'.
    """
    return 0.5 * (1 + np.log(2 * np.pi)) + 0.5 * torch.log(val)

In [ ]:
def compute_loss(true_mean: torch.Tensor, true_logvar: torch.Tensor,
                 model_mean: torch.Tensor, model_logvar: torch.Tensor) -> torch.Tensor:
    """
    Compute the loss based on KL divergence and entropy terms.
    
    Args:
        true_mean, true_logvar: Ground truth posterior parameters.
        model_mean, model_logvar: Predicted parameters.
    
    Returns:
        torch.Tensor: Average loss in bits.
    """
    # KL divergence term
    KL = normal_kl(true_mean, true_logvar, model_mean, model_logvar).float()
    H_start = entropy(betas[0].float()).float()
    beta_full_trajectory = 1.0 - torch.exp(torch.sum(torch.log(alphas)))
    H_end = entropy(beta_full_trajectory.float()).float()
    H_prior = entropy(torch.tensor([1.0], device=device)).float()
    negL_bound = KL * n_steps + H_start - H_end + H_prior
    negL_gauss = entropy(torch.tensor([1.0], device=device)).float()
    negL_diff = negL_bound - negL_gauss
    L_diff_bits = negL_diff / np.log(2.0)
    return L_diff_bits.mean()

#### Training at Random Timesteps

Training a diffusion probabilistic model by randomly selecting timesteps might initially seem counterintuitive. However, it is essential for the model to generalize effectively across all levels of noise encountered during the diffusion process. This training approach, inspired by the [DDIM implementation](https://github.com/ermongroup/ddim), employs a technique known as **antithetic sampling** to enhance stability and convergence.

##### Why Train on Random Timesteps?

The core idea behind diffusion models is to progressively add noise to data during the forward process and then learn to reverse this noise (denoising). To achieve robust performance, the neural network must learn how to accurately denoise samples at **any arbitrary timestep** in the diffusion chain. Training at random timesteps serves two critical purposes:

- **Generalization Across Noise Levels**:  
  Random timestep selection forces the model to learn how to handle samples with varying noise intensities, ranging from nearly clean data (low noise) to almost pure Gaussian noise (high noise). This ensures the model generalizes well throughout the entire denoising trajectory.

- **Comprehensive and Efficient Training**:  
  By randomly sampling timesteps, the model covers all possible scenarios evenly, without needing to explicitly traverse every timestep during each iteration. This provides efficient and balanced exposure across the entire diffusion process.

##### Intuition Behind Antithetic Sampling

The [DDIM implementation](https://github.com/ermongroup/ddim) further refines random timestep training using **antithetic sampling**, which involves pairing timesteps from opposite ends of the diffusion trajectory. Specifically, for a diffusion chain with $T$ timesteps, each chosen timestep $t$ is paired with its "symmetric partner" $T - t - 1$. This approach has several intuitive benefits:

- **Balanced Learning**:  
  By pairing "easy" (low-noise) timesteps with "hard" (high-noise) timesteps, the model simultaneously refines its performance at both ends of the noise spectrum. This ensures the neural network can handle diverse noise scenarios equally well.

- **Reduced Variance and Increased Stability**:  
  Training simultaneously on symmetrically positioned timesteps smooths out gradient variability. This balanced approach helps stabilize training, often accelerating convergence and improving model accuracy.

**Concrete Example:**  
Consider a diffusion model with $T=100$ timesteps:
- Selecting a timestep $t=5$ (minimal noise) automatically pairs it with timestep $T - t - 1 = 94$ (heavy noise).
- Thus, in the same training iteration, the model learns both precise denoising for nearly clean data (step 5) and robust denoising strategies for heavily corrupted samples (step 94).

In [ ]:
def loss_likelihood_bound(model: nn.Module, x_0: torch.Tensor) -> torch.Tensor:
    """
    Compute the variational bound loss at random timesteps.
    
    Args:
        model (nn.Module): The conditional model.
        x_0 (torch.Tensor): Original data batch.
    
    Returns:
        torch.Tensor: Loss value.
    """
    batch_size = x_0.shape[0]
    # Select a random step for each example
    t_half = torch.randint(0, n_steps, (batch_size // 2 + 1,), device=x_0.device)
    t = torch.cat([t_half, n_steps - t_half - 1], dim=0)[:batch_size].long()
    # Perform diffusion for step t
    x_t = q_sample(x_0, t)
    # Compute the true mean and variance
    true_mean, true_logvar = q_posterior_mean_variance(x_0, x_t, t)
    # Infer the mean and variance with our model
    model_mean, model_logvar = p_mean_variance(model, x_t, t)
    # Compute the loss
    return compute_loss(true_mean, true_logvar, model_mean, model_logvar)

### Training Loop for the Original Diffusion Model

The following training loop optimizes the variational bound loss.

In [ ]:
model = ConditionalModel(n_steps).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
dataset = torch.tensor(data.T, dtype=torch.float32, device=device)
batch_size = 128
num_iterations = 5001

for iteration in range(num_iterations):
    permutation = torch.randperm(dataset.size(0), device=device)
    for i in range(0, dataset.size(0), batch_size):
        # Retrieve current batch
        indices = permutation[i:i+batch_size]
        batch_x = dataset[indices]
        # Compute the loss
        loss = loss_likelihood_bound(model, batch_x)
        optimizer.zero_grad()
        loss.backward()
        # Perform gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    if iteration % 1000 == 0:
        print(f"Iteration {iteration}: Loss = {loss.item():.4f}")
        x_seq = p_sample_loop(model, dataset.shape)
        fig, axs = plt.subplots(1, 10, figsize=(28, 3))
        for j in range(1, 11):
            cur_x = x_seq[j * 10].detach().cpu()
            axs[j-1].scatter(cur_x[:, 0], cur_x[:, 1], s=10)
            axs[j-1].set_axis_off()
            axs[j-1].set_title(f"$q(\\mathbf{{x}}_{{{j*100}}})$")
        plt.tight_layout()
        plt.show()

## Denoising Diffusion Probabilistic Models (DDPM)


In their paper, Ho et al. proposed several enhancements to the diffusion model framework, leading to the formulation known as **Denoising Diffusion Probabilistic Models (DDPM)**. These improvements enhance the generative quality and stability of diffusion models.

### New Parameterization: Noise Prediction

A key innovation introduced by Ho et al. is the parameterization of the reverse process mean function directly in terms of the predicted **noise**. Instead of directly predicting the mean, the neural network explicitly predicts the noise component $\epsilon_{\theta}$:

$$
\mathbf{\mu}_{\theta}(\mathbf{x}_t, t) = \frac{1}{\sqrt{\alpha_t}}\left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\epsilon_{\theta}(\mathbf{x}_t, t)\right).
$$

Here:

- $\epsilon_{\theta}(\mathbf{x}_t, t)$ is a neural network that predicts the noise component originally introduced during the forward diffusion process.
- The parameterization in terms of predicted noise significantly simplifies training since the network directly learns the perturbation that must be removed.

### Fixed Variance Schedule

Additionally, the authors propose employing a **fixed variance schedule** for practical reasons. Rather than learning the variance, they define it explicitly as follows:

$$
\sigma_t^2 = \beta_t, \quad \text{or} \quad \sigma_t^2 = \tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}\beta_t
$$

In practice, $\tilde{\beta}_t$ is often chosen due to its better empirical performance. This explicit variance choice simplifies the optimization and stabilizes training.

### Reverse Sampling Procedure in DDPM

The DDPM reverse sampling step, using the noise prediction parameterization and fixed variance, is now explicitly defined as:

$$
\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(\mathbf{x}_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}}\epsilon_{\theta}(\mathbf{x}_t, t)\right) + \sigma_t \mathbf{z}, \quad \mathbf{z} \sim \mathcal{N}(0, \mathbf{I})
$$

Here:

- The reverse step directly "subtracts" the predicted noise, scaled appropriately by factors derived from the forward diffusion schedule ($\alpha_t$, $\bar{\alpha}_t$).
- $\sigma_t \mathbf{z}$ explicitly injects Gaussian randomness, following the predefined variance schedule.

This reformulation clearly separates the roles of deterministic denoising (the neural network's prediction of $\epsilon_\theta$) and stochastic exploration (via the Gaussian term), improving both model interpretability and sampling efficiency.


In [ ]:
class ConditionalModelDDPM(nn.Module):
    """
    DDPM conditional model that predicts the noise component.
    """
    def __init__(self, n_steps: int):
        super(ConditionalModelDDPM, self).__init__()
        self.lin1 = ConditionalLinear(2, 128, n_steps)
        self.lin2 = ConditionalLinear(128, 128, n_steps)
        self.lin3 = ConditionalLinear(128, 128, n_steps)
        self.lin4 = nn.Linear(128, 2)  # Predict noise
        
    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        x = F.softplus(self.lin1(x, t))
        x = F.softplus(self.lin2(x, t))
        x = F.softplus(self.lin3(x, t))
        return self.lin4(x)

In [ ]:
def p_sample_ddpm(model: nn.Module, x: torch.Tensor, t: int) -> torch.Tensor:
    """
    Perform one reverse diffusion step for DDPM.
    
    Args:
        model (nn.Module): DDPM conditional model.
        x (torch.Tensor): Current sample at timestep t.
        t (int): Current timestep.
    
    Returns:
        torch.Tensor: Sampled x_{t-1}.
    """
    t_tensor = torch.tensor([t], device=x.device).long()
    alpha_t = extract(alphas, t_tensor, x)
    one_minus_alpha_bar = extract(one_minus_alphas_bar_sqrt, t_tensor, x)
    # Model output
    eps_theta = model(x, t_tensor)
    # Factor to the model output
    eps_factor = (1 - alpha_t) / one_minus_alpha_bar
    mean = (1 / torch.sqrt(alpha_t)) * (x - eps_factor * eps_theta)
    # Fixed sigma
    sigma_t = torch.sqrt(extract(betas, t_tensor, x))
    noise = torch.randn_like(x)
    return mean + sigma_t * noise

Notably, the forward process posterior distributions are analytically tractable when conditioned on the original data point $\mathbf{x}_{0}$. Precisely, the forward diffusion step is defined by:

$$
q(\mathbf{x}_{t}\mid\mathbf{x}_{t-1}) = \mathcal{N}\left(\mathbf{x}_{t}; \sqrt{1 - \beta_{t}}\,\mathbf{x}_{t-1},\, \beta_{t}\mathbf{I}\right).
$$

Since the forward diffusion process consists of a sequence of Gaussian transitions, the posterior distribution of the previous timestep, conditioned explicitly on the current latent state $\mathbf{x}_{t}$ and the initial data $\mathbf{x}_{0}$, is also Gaussian:

$$
q(\mathbf{x}_{t-1}\mid\mathbf{x}_{t},\mathbf{x}_{0}) = \mathcal{N}\left(\mathbf{x}_{t-1};\,\tilde{\mu}_{t}(\mathbf{x}_{t},\mathbf{x}_{0}),\,\tilde{\beta}_{t}\mathbf{I}\right),
$$

where the posterior mean $\tilde{\mu}_{t}$ and posterior variance $\tilde{\beta}_{t}$ have exact closed-form expressions:

- **Posterior mean**:
$$
\tilde{\mu}_{t}(\mathbf{x}_{t},\mathbf{x}_{0}) = \frac{\sqrt{\bar{\alpha}_{t-1}}\,\beta_{t}}{1 - \bar{\alpha}_{t}}\mathbf{x}_{0} + \frac{\sqrt{\alpha_{t}}\,(1 - \bar{\alpha}_{t-1})}{1 - \bar{\alpha}_{t}}\mathbf{x}_{t}.
$$

- **Posterior variance**:
$$
\tilde{\beta}_{t} = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_{t}}\beta_{t}.
$$

These equations explicitly connect the latent state at timestep $t$ and the original data $\mathbf{x}_{0}$, clearly indicating how much information from the original data and the current noisy state contributes to reconstructing the previous timestep.


### Training in DDPM

In **Denoising Diffusion Probabilistic Models (DDPM)**, training is performed by optimizing the Evidence Lower Bound (ELBO) of the data log-likelihood. A key insight from Ho et al. was to rewrite the ELBO as an explicit sum of analytically tractable KL divergences, significantly improving training stability through variance reduction.

Specifically, the loss function $\mathcal{L}$ can be expressed clearly as:

$$
\mathcal{L}(\theta) = \mathbb{E}_{q}\left[ \mathcal{L}_{T} + \sum_{t=2}^{T}\mathcal{L}_{t-1} + \mathcal{L}_{0} \right],
$$

with each term explicitly defined as follows:

#### 1. Prior Matching Loss ($\mathcal{L}_{T}$):
$$
\mathcal{L}_{T} = D_{KL}\left(q(\mathbf{x}_{T}\mid\mathbf{x}_{0})\,\Vert\,p(\mathbf{x}_{T})\right).
$$

This term quantifies the difference between the final state of the forward diffusion (typically Gaussian noise) and the chosen Gaussian prior $p(\mathbf{x}_{T})$. Minimizing this term ensures that the forward diffusion process correctly maps the original data to the simple tractable prior distribution.

#### 2. Reverse Process Loss ($\mathcal{L}_{t-1}$):
$$
\mathcal{L}_{t-1} = D_{KL}\left(q(\mathbf{x}_{t-1}\mid\mathbf{x}_{t},\mathbf{x}_{0})\,\Vert\,p_{\theta}(\mathbf{x}_{t-1}\mid\mathbf{x}_{t})\right).
$$

This is the core loss component that trains the reverse (denoising) neural network to approximate the exact forward posterior distribution. It directly penalizes the deviation of the neural model predictions from the true analytically-computed posteriors, enabling efficient training.

#### 3. Reconstruction Loss ($\mathcal{L}_{0}$):
$$
\mathcal{L}_{0} = -\log p_{\theta}(\mathbf{x}_{0}\mid\mathbf{x}_{1}).
$$

This term encourages the model to accurately reconstruct the original data $\mathbf{x}_0$ from its slightly noisy version $\mathbf{x}_1$. Minimizing this loss ensures high-quality reconstructions, directly connecting the latent representations to the observed data distribution.

#### Analytical Tractability and Variance Reduction

Importantly, all KL divergences in the above formulation compare Gaussian distributions, enabling closed-form analytical solutions. This analytical tractability greatly reduces the variance during training and significantly accelerates convergence:

- **Closed-form KL divergence between two Gaussians:**  
  Given two Gaussian distributions, $q=\mathcal{N}(\mu_q,\Sigma_q)$ and $p=\mathcal{N}(\mu_p,\Sigma_p)$, the KL divergence is analytically computed as:
  $$
  D_{KL}(q||p)=\frac{1}{2}\left(\text{tr}\left(\Sigma_p^{-1}\Sigma_q\right)+(\mu_p-\mu_q)^\top\Sigma_p^{-1}(\mu_p-\mu_q)-d+\log\frac{|\Sigma_p|}{|\Sigma_q|}\right),
  $$

  where $d$ is the dimensionality of the data. Because variances are often diagonal and fixed, this expression simplifies significantly in practice, further reducing computational cost.

In [ ]:
def approx_standard_normal_cdf(x: torch.Tensor) -> torch.Tensor:
    """
    Approximate the standard normal CDF using a tanh-based function.
    """
    return 0.5 * (1.0 + torch.tanh(torch.sqrt(2.0 / np.pi) * (x + 0.044715 * torch.pow(x, 3))))

def discretized_gaussian_log_likelihood(x: torch.Tensor, means: torch.Tensor, log_scales: torch.Tensor) -> torch.Tensor:
    """
    Compute log-likelihood for discretized Gaussian data.
    
    Assumes data is integers [0, 255] scaled to [-1, 1].
    """
    centered_x = x - means
    inv_stdv = torch.exp(-log_scales)
    plus_in = inv_stdv * (centered_x + 1.0 / 255.0)
    cdf_plus = approx_standard_normal_cdf(plus_in)
    min_in = inv_stdv * (centered_x - 1.0 / 255.0)
    cdf_min = approx_standard_normal_cdf(min_in)
    log_cdf_plus = torch.log(torch.clamp(cdf_plus, min=1e-12))
    log_one_minus_cdf_min = torch.log(torch.clamp(1 - cdf_min, min=1e-12))
    cdf_delta = cdf_plus - cdf_min
    log_probs = torch.where(x < -0.999, log_cdf_plus,
                 torch.where(x > 0.999, log_one_minus_cdf_min,
                 torch.log(torch.clamp(cdf_delta, min=1e-12))))
    return log_probs

This leads to a new loss function as implemented in the following (note that this objective does not provide large change to the optimization itself).

In [ ]:
def loss_variational(model: nn.Module, x_0: torch.Tensor) -> torch.Tensor:
    """
    Compute the variational loss for DDPM.
    
    Args:
        model (nn.Module): DDPM conditional model.
        x_0 (torch.Tensor): Original data batch.
    
    Returns:
        torch.Tensor: Loss value.
    """
    batch_size = x_0.shape[0]
    # Select a random step for each example
    t_half = torch.randint(0, n_steps, (batch_size // 2 + 1,), device=x_0.device)
    t = torch.cat([t_half, n_steps - t_half - 1], dim=0)[:batch_size].long()
    # Perform diffusion for step t
    x_t = q_sample(x_0, t)
    # Compute the true mean and variance
    true_mean, true_logvar = q_posterior_mean_variance(x_0, x_t, t)
    # Infer the mean and variance with our model
    model_mean, model_logvar = p_mean_variance(model, x_t, t)
    # Compute the KL loss
    kl = normal_kl(true_mean, true_logvar, model_mean, model_logvar)
    kl = torch.mean(kl.view(batch_size, -1), dim=1) / np.log(2.0)
    # NLL of the decoder
    decoder_nll = -discretized_gaussian_log_likelihood(x_0, means=model_mean, log_scales=0.5 * model_logvar)
    decoder_nll = torch.mean(decoder_nll.view(batch_size, -1), dim=1) / np.log(2.0)
    # At the first timestep return the decoder NLL, otherwise return KL(q(x_{t-1}|x_t,x_0) || p(x_{t-1}|x_t))
    output = torch.where(t == 0, decoder_nll, kl)
    return output.mean()

#### Explicitly Simplified Practical Training Objective

In practice, DDPMs often simplify the above objective further by using the noise-prediction parameterization. The training objective then effectively reduces to a simplified noise-matching loss:

$$
\mathcal{L}_{\text{simple}}(\theta)=\mathbb{E}_{t,\mathbf{x}_{0},\epsilon}\left[\|\epsilon-\epsilon_{\theta}(\sqrt{\bar{\alpha}_t}\mathbf{x}_0+\sqrt{1-\bar{\alpha}_t}\epsilon,t)\|^2\right],\quad \epsilon\sim\mathcal{N}(0,\mathbf{I}),
$$
which corresponds directly to minimizing these KL divergences under appropriate parameterizations. We can see that this objective now very closely ressemble the denoising score matching formulation.

In [ ]:
def noise_estimation_loss(model: nn.Module, x_0: torch.Tensor) -> torch.Tensor:
    """
    Compute the simplified noise estimation loss for DDPM.
    
    Args:
        model (nn.Module): DDPM conditional model.
        x_0 (torch.Tensor): Original data batch.
    
    Returns:
        torch.Tensor: Mean squared error between the true noise and the model's prediction.
    """
    batch_size = x_0.shape[0]
    # Select a random step for each example
    t_half = torch.randint(0, n_steps, (batch_size // 2 + 1,), device=x_0.device)
    t = torch.cat([t_half, n_steps - t_half - 1], dim=0)[:batch_size].long()
    # x0 multiplier
    a = extract(alphas_bar_sqrt, t, x_0)
    # eps multiplier
    am1 = extract(one_minus_alphas_bar_sqrt, t, x_0)
    noise = torch.randn_like(x_0)
    # model input
    x = a * x_0 + am1 * noise
    predicted_noise = model(x, t)
    return (noise - predicted_noise).square().mean()

#### Stabilizing training with Exponential Moving Average (EMA)

This idea is found in most of the implementations, which allows to implement a form of _model momentum_. Instead of directly updating the weights of the model, we keep a copy of the previous values of the weights, and then update a weighted mean between the previous and new version of the weights. Here, we reuse the implementation proposed in the [DDIM repository](https://github.com/ermongroup/ddim).

In [ ]:
class EMA:
    """
    Exponential Moving Average (EMA) for model parameter stabilization.
    """
    def __init__(self, mu: float = 0.999):
        self.mu = mu
        self.shadow = {}

    def register(self, module: nn.Module) -> None:
        for name, param in module.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, module: nn.Module) -> None:
        for name, param in module.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (1.0 - self.mu) * param.data + self.mu * self.shadow[name]

    def apply_shadow(self, module: nn.Module) -> None:
        """
        Replace the module parameters with the EMA (shadow) parameters.
        """
        for name, param in module.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.shadow[name])

### DDPM Training Loop with EMA

The training loop below uses the simplified noise estimation loss and applies EMA to stabilize training.


In [ ]:
def p_sample_loop_ddpm(model: nn.Module, shape: tuple) -> list:
    """
    Generate a full sample by iteratively applying the DDPM reverse diffusion process.
    
    Args:
        model (nn.Module): The DDPM conditional model.
        shape (tuple): Shape of the initial noise sample.
    
    Returns:
        List[torch.Tensor]: Sequence of samples from x_T down to x_0.
    """
    with torch.no_grad():
        cur_x = torch.randn(shape, device=device)
        x_seq = [cur_x]
        for i in reversed(range(n_steps)):
            cur_x = p_sample_ddpm(model, cur_x, i)
            x_seq.append(cur_x)
    return x_seq

In [ ]:
model_ddpm = ConditionalModelDDPM(n_steps).to(device)
optimizer = optim.Adam(model_ddpm.parameters(), lr=1e-3)
dataset = torch.tensor(data.T, dtype=torch.float32, device=device)
ema = EMA(mu=0.9)
ema.register(model_ddpm)
batch_size = 128
num_iterations_ddpm = 1000

for iteration in range(num_iterations_ddpm):
    permutation = torch.randperm(dataset.size(0), device=device)
    for i in range(0, dataset.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_x = dataset[indices]
        loss = noise_estimation_loss(model_ddpm, batch_x)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ddpm.parameters(), 1.0)
        optimizer.step()
        ema.update(model_ddpm)
    if iteration % 100 == 0:
        print(f"DDPM Iteration {iteration}: Loss = {loss.item():.4f}")
        x_seq = p_sample_loop_ddpm(model_ddpm, dataset.shape)
        fig, axs = plt.subplots(1, 10, figsize=(28, 3))
        for j in range(1, 11):
            cur_x = x_seq[j * 10].detach().cpu()
            axs[j-1].scatter(cur_x[:, 0], cur_x[:, 1], s=10)
            axs[j-1].set_title(f"$q(\\mathbf{{x}}_{{{j*100}}})$")
            axs[j-1].set_axis_off()
        plt.tight_layout()
        plt.show()